## Data Analysis

**【Data Science Project】 Building a Machine Learning Pipeline for a Predictive Car Price Model with PySpark**

In [138]:
import os

# point java home to actual conda package reference
os.environ["JAVA_HOME"] = "/Users/andreasliistro/mambaforge/pkgs/openjdk-22.0.1-hbeb2e11_0/lib/jvm"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import isnan, when, count, col, lit
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

# init spark session
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [139]:
# load data into spark
data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles.csv", header=True, inferSchema=True)
data.show()

+----------+--------------------+--------------------+--------------------+-----+----+------------+-----+---------+---------+----+--------+------------+------------+----+-----+----+----+-----------+---------+-----------+------+-----+----+----+------------+
|        id|                 url|              region|          region_url|price|year|manufacturer|model|condition|cylinders|fuel|odometer|title_status|transmission| VIN|drive|size|type|paint_color|image_url|description|county|state| lat|long|posting_date|
+----------+--------------------+--------------------+--------------------+-----+----+------------+-----+---------+---------+----+--------+------------+------------+----+-----+----+----+-----------+---------+-----------+------+-----+----+----+------------+
|7222695916|https://prescott....|            prescott|https://prescott....| 6000|NULL|        NULL| NULL|     NULL|     NULL|NULL|    NULL|        NULL|        NULL|NULL| NULL|NULL|NULL|       NULL|     NULL|       NULL|  NULL|  

In [140]:
# show schema
data.printSchema()

root
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: string (nullable = true)
 |-- year: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: string (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- posting_date: string (nu

In [141]:
# show statisctics
data.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
id,441802,7.311486634224333E9,4473170.412558007,1985 Mazda RX-7 Gs 08e147c95bc740b796...,7317101084
url,431918,1406.3489361702127,919.4004721287523,2005 Subaru Legacy Wagon Outback 2.5i Ma...,https://zanesville.craigslist.org/cto/d/zanesv...
region,434901,1427.686274509804,808.3057161506485,1500,zanesville / cambridge
region_url,435269,157.6067265494309,474.66485537053,1500,wa
price,435356,74186.53764402741,1.2099967417235982E7,,wa
year,433912,2005.5686682071962,112.0514923702533,$489 Doc charge (clerical processing / docume...,wa
manufacturer,412865,744.1736842578244,1029.7129298116581,2010,volvo
model,424296,1914.5716864965718,5327.649155487644,2007,🔥GMC Sierra 1500 SLE🔥 4X4 🔥
condition,254659,549.0233284319527,959.2700455051274,2006,salvage


In [142]:
# drop columns not interested in
data = data.drop("url",
                 "region_url", 
                 "image_url", 
                 "vin", 
                 "lat", 
                 "long",
                 "region"
                 )

# handle "description" differently


In [143]:
# analyse missing values
data.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in data.columns]).show()

# analyis missing values in percentage
data.select([(count(when(isnan(c) | col(c).isNull(), c)) / count(lit(1)) * 100).alias(c) for c in data.columns]).show()


+---+-----+----+------------+-----+---------+---------+-----+--------+------------+------------+------+------+------+-----------+-----------+------+-----+------------+
| id|price|year|manufacturer|model|condition|cylinders| fuel|odometer|title_status|transmission| drive|  size|  type|paint_color|description|county|state|posting_date|
+---+-----+----+------------+-----+---------+---------+-----+--------+------------+------------+------+------+------+-----------+-----------+------+-----+------------+
|  0| 6446|7890|       28937|17506|   187143|   190798|16344|   17801|       21618|       15932|144178|319997|106892|     143839|      13733|382283|23077|       22477|
+---+-----+----+------------+-----+---------+---------+-----+--------+------------+------------+------+------+------+-----------+-----------+------+-----+------------+



+---+------------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+----------------+-----------------+-----------------+----------------+-----------------+-----------------+-----------------+
| id|             price|             year|     manufacturer|             model|         condition|        cylinders|              fuel|         odometer|     title_status|     transmission|            drive|            size|             type|      paint_color|     description|           county|            state|     posting_date|
+---+------------------+-----------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+-----------------+-----------------+----------------+-----------------+-----------------+----------------+-----------------+-----------------+-----------------+
|0.0

Checking empty / null values shows following columns with more than 1/3 empty values, which can be removed
- cylinders
- county

Following additional columns contain high number of empty values but identified as important columns, therefore can't be removed
- paint_color
- condition
- drive
- size

In [144]:
# delete columns with larger number of missing values
data = data.drop("cylinders", "county")


In [145]:
# delete rows with missing price
data = data.na.drop(subset=["price"])

# convert "price" to double and remove invalid doubles values
data = data.withColumn("price", data["price"].cast("double"))

In [146]:
# leave nulls in but replace unreal values with null
data = data.withColumn("year", when(col("year") < 1900, None).otherwise(col("year")))
data = data.withColumn("year", when(col("year") > 2025, None).otherwise(col("year")))

# convert "year" to double
data = data.withColumn("year", data["year"].cast("double"))

In [147]:
# feature filtering for manufacturer
manufacture_counts = data.groupBy("manufacturer").count().orderBy(col("count").desc())

# use only more than 10 occurences
supported_manufacture = manufacture_counts.filter(col("count") < 10)

# filter data to only include supported manufactures
data = data.join(supported_manufacture, "manufacturer", "left_anti")

# show new manufacturer counts
data.groupBy("manufacturer").count().orderBy(col("count").desc()).show()

+-------------+-----+
| manufacturer|count|
+-------------+-----+
|         ford|70985|
|    chevrolet|55064|
|       toyota|34202|
|         NULL|22578|
|        honda|21269|
|       nissan|19067|
|         jeep|19014|
|          ram|18342|
|          gmc|16785|
|          bmw|14699|
|        dodge|13707|
|mercedes-benz|11817|
|      hyundai|10338|
|       subaru| 9495|
|   volkswagen| 9345|
|          kia| 8457|
|        lexus| 8200|
|         audi| 7573|
|     cadillac| 6953|
|     chrysler| 6031|
+-------------+-----+
only showing top 20 rows



In [148]:
# feature filtering for model
model_counts = data.groupBy("model").count().orderBy(col("count").desc())

# remove models with only one occurence
supported_model = model_counts.filter(col("count") < 2)

# filter data to only include supported models
data = data.join(supported_model, "model", "left_anti")

# replace Null with unknown
data = data.fillna("unknown", subset=["model"])

# show new model counts
data.groupBy("model").count().orderBy(col("count").desc()).show()

+--------------+-----+
|         model|count|
+--------------+-----+
|       unknown|10299|
|         f-150| 8009|
|silverado 1500| 5140|
|          1500| 4211|
|         camry| 3135|
|     silverado| 3023|
|        accord| 2969|
|      wrangler| 2848|
|         civic| 2799|
|        altima| 2779|
|        escape| 2746|
|          2500| 2687|
|        tacoma| 2582|
|      explorer| 2499|
|grand cherokee| 2489|
|       corolla| 2241|
|       mustang| 2225|
|        fusion| 1979|
|       equinox| 1972|
|          cr-v| 1930|
+--------------+-----+
only showing top 20 rows



In [149]:
# clean condition data
condition_counts = data.groupBy("condition").count().orderBy(col("count").desc())

supported_condition = [
  "good",
  "excellent",
  "like new",
  "fair",
  "new",
  "salvage",
  " preowned cars",
  "physical damage insurance",
  "unknown",
]

# replace Null with unknown
data = data.fillna("unknown", subset=["condition"])

# replace all other values with other
data = data.withColumn("condition", when(col("condition").isin(supported_condition), col("condition")).otherwise("other"))

# show new values for condition
data.groupBy("condition").count().orderBy(col("count").desc()).show(20)

+--------------+------+
|     condition| count|
+--------------+------+
|       unknown|174343|
|          good|118147|
|     excellent| 96795|
|      like new| 19878|
|          fair|  6152|
|           new|  1169|
|         other|  1137|
| preowned cars|   695|
|       salvage|   559|
+--------------+------+



In [150]:
# clean fuel data
fuel_counts = data.groupBy("fuel").count().orderBy(col("count").desc())

# replace Null with unknown
data = data.fillna("unknown", subset=["fuel"])

supported_fuel = [
  "gas",
  "diesel",
  "other",
  "hybrid",
  "electric",
  "unknown",
]

# replace all other values with other
data = data.withColumn("fuel", when(col("fuel").isin(supported_fuel), col("fuel")).otherwise("other"))

# show new values for fuel
data.groupBy("fuel").count().orderBy(col("count").desc()).show(20)

+--------+------+
|    fuel| count|
+--------+------+
|     gas|343869|
|   other| 31761|
|  diesel| 28316|
| unknown|  8335|
|  hybrid|  5001|
|electric|  1593|
+--------+------+



In [151]:
# odometer cleaning
data = data.withColumn("odometer", when(col("odometer") < 0, None).otherwise(col("odometer")))

# convert "odometer" to double
data = data.withColumn("odometer", data["odometer"].cast("double"))

In [152]:
# transmission cleaning
transmission_counts = data.groupBy("transmission").count().orderBy(col("count").desc())

supported_transmission = [
  "automatic",
  "manual",
  "other",
  "unknown",
]

# replace Null with unknown
data = data.fillna("unknown", subset=["transmission"])

# replace all other values with other
data = data.withColumn("transmission", when(col("transmission").isin(supported_transmission), col("transmission")).otherwise("other"))

# show new values for transmission
data.groupBy("transmission").count().orderBy(col("count").desc()).show(20)

+------------+------+
|transmission| count|
+------------+------+
|   automatic|323938|
|       other| 63740|
|      manual| 22921|
|     unknown|  8276|
+------------+------+



In [153]:
# drive cleaning
drive_counts = data.groupBy("drive").count().orderBy(col("count").desc())

supported_drive = [
  "fwd",
  "4wd",
  "rwd",
  "unknown",
]

# replace Null with unknown
data = data.fillna("unknown", subset=["drive"])

# replace all other values with other
data = data.withColumn("drive", when(col("drive").isin(supported_drive), col("drive")).otherwise("other"))

# show new values for drive
data.groupBy("drive").count().orderBy(col("count").desc()).show(20)

+-------+------+
|  drive| count|
+-------+------+
|unknown|132237|
|    4wd|127212|
|    fwd|102671|
|    rwd| 55451|
|  other|  1304|
+-------+------+



In [154]:
# size cleaning
size_counts = data.groupBy("size").count().orderBy(col("count").desc())

supported_size = [
  "full-size",
  "mid-size",
  "compact",
  "sub-compact",
  "unknown",
]

# replace Null with unknown
data = data.fillna("unknown", subset=["size"])

# replace all other values with other
data = data.withColumn("size", when(col("size").isin(supported_size), col("size")).otherwise("other"))

# show new values for size
data.groupBy("size").count().orderBy(col("count").desc()).show(20)

+-----------+------+
|       size| count|
+-----------+------+
|    unknown|302983|
|  full-size| 60140|
|   mid-size| 32846|
|    compact| 18563|
|sub-compact|  3064|
|      other|  1279|
+-----------+------+



In [155]:
# state cleaning
state_counts = data.groupBy("state").count().orderBy(col("count").desc())

# list of all US states shortcodes
supported_states = [
  "al", "ak", "az", "ar", "ca", "co", "ct", "de", "fl", "ga", "hi", "id", "il", "in", "ia", "ks", "ky", "la", "me", "md", "ma", "mi", "mn", "ms", "mo", "mt", "ne", "nv", "nh", "nj", "nm", "ny", "nc", "nd", "oh", "ok", "or", "pa", "ri", "sc", "sd", "tn", "tx", "ut", "vt", "va", "wa", "wv", "wi", "wy"
]

# replace Null with unknown
data = data.fillna("unknown", subset=["state"])

# replace all other values with other
data = data.withColumn("state", when(col("state").isin(supported_states), col("state")).otherwise("other"))

# show new values for state
data.groupBy("state").count().orderBy(col("count").desc()).show(20)

+-----+-----+
|state|count|
+-----+-----+
|other|72739|
|   ca|41980|
|   fl|23482|
|   tx|19433|
|   ny|16930|
|   oh|15235|
|   mi|14566|
|   or|11846|
|   pa|11790|
|   nc|11731|
|   wi|10009|
|   il| 9257|
|   tn| 9234|
|   nj| 8873|
|   va| 8523|
|   co| 8278|
|   wa| 7124|
|   az| 7123|
|   ia| 7114|
|   ma| 7046|
+-----+-----+
only showing top 20 rows



In [156]:
# paint_color cleaning
paint_color_counts = data.groupBy("paint_color").count().orderBy(col("count").desc())

supported_paint_color = [
  "white",
  "black",
  "silver",
  "blue",
  "red",
  "grey",
  "green",
  "brown",
  "yellow",
  "custom",
  "orange",
  "purple",
  "unknown",
]

# replace Null with unknown
data = data.fillna("unknown", subset=["paint_color"])

# replace all other values with other
data = data.withColumn("paint_color", when(col("paint_color").isin(supported_paint_color), col("paint_color")).otherwise("other"))

# show new values for paint_color
data.groupBy("paint_color").count().orderBy(col("count").desc()).show(20)

+-----------+------+
|paint_color| count|
+-----------+------+
|    unknown|131290|
|      white| 76503|
|      black| 61009|
|     silver| 41684|
|       blue| 30153|
|        red| 29419|
|       grey| 23431|
|      green|  6934|
|     custom|  6383|
|      brown|  6299|
|     yellow|  1970|
|     orange|  1884|
|      other|  1279|
|     purple|   637|
+-----------+------+



In [157]:
# show new schema
data.printSchema()

root
 |-- model: string (nullable = false)
 |-- manufacturer: string (nullable = true)
 |-- id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- condition: string (nullable = false)
 |-- fuel: string (nullable = false)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = false)
 |-- drive: string (nullable = false)
 |-- size: string (nullable = false)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = false)
 |-- description: string (nullable = true)
 |-- state: string (nullable = false)
 |-- posting_date: string (nullable = true)



In [158]:
# show statistics
data.describe().toPandas().transpose()

,0,1,2,3,4
summary,count,mean,stddev,min,max
model,418875,1787.3356730426565,3000.226205206505,2007,♿ vmi
manufacturer,400570,2013.0827067669172,2.4357250295035895,2010,volvo
id,418875,7.311451577137132E9,4476911.557922282,1985 Mazda RX-7 Gs 08e147c95bc740b796...,7317098055
price,416601,75255.42155229673,1.2325434572704755E7,-158.030906,3.736928711E9
year,410678,2011.4946600499661,8.966693643802259,1900.0,2022.0
condition,418875,None,None,preowned cars,unknown
fuel,418875,None,None,diesel,unknown
odometer,407358,96725.64597811851,197202.21309065283,0.0,1.0E7
title_status,405385,324.46937506426735,213.5919738551412,150,salvage


In [160]:
# safe data as new csv
data.write.csv("ML-pipeLine-car-prediction/data/vehicles_cleaned.csv", header=True)